In [7]:
import pandas as pd
import numpy as np
import os
import pickle
from typing import Dict, Any

# --- 모델 하이퍼파라미터 (규칙) 정의 ---

# 1. 서프라이즈 필터 (ARIMA Z-Score)
Z_SCORE_THRESHOLD = 2.0 

# 2. GICS 필터 (최적 Sector)
MOST_SENSITIVE_GICS_SECTORS = [35.0] 

# 3. 기술적 지표 필터
MOMENTUM_THRESHOLD = 0.08      # 20일 모멘텀 8% 이상
VOLUME_RATIO_THRESHOLD = 2.0   # 거래량 비율 2.0배 이상

# 4. RSI 필터
RSI_LOWER_BOUND = 50.0 
RSI_UPPER_BOUND = 75.0 

# 모델 메타데이터
MODEL_FILE_NAME = "model_ver5.pkl"
OUT_DIR = "../../output/model" 
os.makedirs(OUT_DIR, exist_ok=True)

In [8]:
def get_investment_decision_ver5(
    surprise_z: float, 
    gics_code: float, 
    momentum_rate: float, 
    volume_ratio: float,
    rsi: float
) -> Dict[str, Any]:
    """
    최종 통합 하이브리드 모델 (Model Version 5).
    5가지 조건을 모두 충족할 때만 BUY 결정.
    """
    
    # 1. Decision Logic Check (ALL-IN Strategy)
    is_surprise_ok = (surprise_z > Z_SCORE_THRESHOLD)
    is_gics_ok = (gics_code in MOST_SENSITIVE_GICS_SECTORS)
    is_momentum_ok = (momentum_rate >= MOMENTUM_THRESHOLD)
    is_volume_ok = (volume_ratio >= VOLUME_RATIO_THRESHOLD)
    is_rsi_ok = (rsi > RSI_LOWER_BOUND) and (rsi <= RSI_UPPER_BOUND)
    
    
    # 2. Final Decision 
    if is_surprise_ok and is_gics_ok and is_momentum_ok and is_volume_ok and is_rsi_ok:
        decision = 'BUY'
        reason = "ALL 5 CONDITIONS MET: Optimal GICS, Strong Surprise, High Momentum/Volume, and Healthy RSI Range."
    else:
        decision = 'HOLD'
        
        # Output Reason for HOLD decision
        missing_filters = []
        if not is_surprise_ok: missing_filters.append("Surprise Z-Score")
        if not is_gics_ok: missing_filters.append("GICS Sector")
        if not is_momentum_ok: missing_filters.append("Momentum Rate")
        if not is_volume_ok: missing_filters.append("Volume Ratio")
        if not is_rsi_ok: missing_filters.append("RSI (50-75)")
        
        reason = f"HOLD. Missing Filters: {', '.join(missing_filters)}"


    # 3. Output Assembly (Expected Return must be looked up separately by the API)
    return {
        'decision': decision,
        'expected_20d_return_note': "API must pull Avg_Return_Post_20D for GICS 35.0.",
        'reason': reason
    }

In [9]:
# --- 모델 규칙 저장 (.pkl) ---
model_rules = {
    'model_name': 'Final_5Factor_Hybrid_V5',
    'z_threshold': Z_SCORE_THRESHOLD,
    'gics_sectors': MOST_SENSITIVE_GICS_SECTORS, 
    'momentum_threshold': MOMENTUM_THRESHOLD,
    'volume_threshold': VOLUME_RATIO_THRESHOLD,
    'rsi_range': (RSI_LOWER_BOUND, RSI_UPPER_BOUND),
    'version': 'v5.0 (5-Factor Hybrid)'
}

model_path = os.path.join(OUT_DIR, MODEL_FILE_NAME)
with open(model_path, 'wb') as f:
    pickle.dump(model_rules, f)
    
print(f"\n[OK] 최종 모델 Version 5 규칙 저장 완료: {model_path}")


# --- 모델 테스트 (API 호출 시뮬레이션) ---
print("\n" + "="*80)
print("FINAL MODEL (VERSION 5) TEST CASES")
print("="*80)

# Case 1: ALL PASS (Expected BUY)
test_case_1 = get_investment_decision_ver5(
    surprise_z=2.5,     # PASS
    gics_code=35.0,     # PASS
    momentum_rate=0.15, # PASS
    volume_ratio=2.5,   # PASS
    rsi=65.0            # PASS
)
print("Case 1 (BUY):", test_case_1['decision'], "| Reason:", test_case_1['reason'])

# Case 2: RSI FAIL (Overbought, Expected HOLD)
test_case_2 = get_investment_decision_ver5(
    surprise_z=3.5, 
    gics_code=35.0,
    momentum_rate=0.15,
    volume_ratio=2.5,
    rsi=85.0           # FAIL
)
print("Case 2 (RSI FAIL - HOLD):", test_case_2['decision'], "| Reason:", test_case_2['reason'])

print("="*80)


[OK] 최종 모델 Version 5 규칙 저장 완료: ../../output/model/model_ver5.pkl

FINAL MODEL (VERSION 5) TEST CASES
Case 1 (BUY): BUY | Reason: ALL 5 CONDITIONS MET: Optimal GICS, Strong Surprise, High Momentum/Volume, and Healthy RSI Range.
Case 2 (RSI FAIL - HOLD): HOLD | Reason: HOLD. Missing Filters: RSI (50-75)


In [ ]:
import pandas as pd
import numpy as np
import os

# --- 모델 규칙 정의 (V5 기준) ---
Z_SCORE_THRESHOLD = 2.0 
MOST_SENSITIVE_GICS_SECTORS = [35.0] 
MOMENTUM_THRESHOLD = 0.08
VOLUME_RATIO_THRESHOLD = 2.0
RSI_LOWER_BOUND = 50.0 
RSI_UPPER_BOUND = 75.0 

# 로드할 컬럼: 모든 분석 요소와 수익률
REQUIRED_COLS = ['symbol', 'date', 'surprise_z', 'gics_code', 'momentum_rate', 'volume_ratio', 'rsi', 'return_post_1d', 'return_post_2d']
RETURN_COLS_SUMMARY = ['return_post_1d', 'return_post_2d'] # 출력할 수익률 컬럼

# --- 경로 설정 ---
VENDOR_ANALYSIS_PATH = "../../output/problem2_vendor/problem2_vendor_analysis_base.csv"
OUT_DIR = "../../output/model" 
os.makedirs(OUT_DIR, exist_ok=True) 


# --- 1. 데이터 로드 및 V5 규칙 적용 ---
print("-> 1. 데이터 로드 및 모델 Version 5 규칙 적용...")

try:
    # 4가지 필터 변수와 수익률 컬럼을 모두 로드
    df_analysis = pd.read_csv(
        VENDOR_ANALYSIS_PATH, 
        usecols=REQUIRED_COLS,
        dtype={col: np.float32 for col in REQUIRED_COLS if col not in ['symbol', 'date']}
    ).dropna(subset=['surprise_z', 'gics_code', 'momentum_rate', 'volume_ratio', 'rsi', 'return_post_1d'])

    # 2. V5 통합 규칙 적용
    is_surprise_ok = (df_analysis['surprise_z'] > Z_SCORE_THRESHOLD)
    is_gics_ok = (df_analysis['gics_code'].isin(MOST_SENSITIVE_GICS_SECTORS))
    is_momentum_ok = (df_analysis['momentum_rate'] >= MOMENTUM_THRESHOLD)
    is_volume_ok = (df_analysis['volume_ratio'] >= VOLUME_RATIO_THRESHOLD)
    is_rsi_ok = (df_analysis['rsi'] > RSI_LOWER_BOUND) & (df_analysis['rsi'] <= RSI_UPPER_BOUND)
    
    # Final Decision: 5가지 조건 모두 충족해야 BUY
    is_buy_signal = is_surprise_ok & is_gics_ok & is_momentum_ok & is_volume_ok & is_rsi_ok
    df_analysis['decision'] = np.where(is_buy_signal, 'BUY', 'HOLD')
    
    # 시그널 발생 행만 필터링 (BUY Only)
    df_signals = df_analysis[df_analysis['decision'] == 'BUY'].copy()
    
    if df_signals.empty:
        print("분석: 최종 5-Factor 필터링 결과, BUY 시그널이 발생하지 않아 검증을 수행할 수 없습니다.")
        exit()

except FileNotFoundError as e:
    print(f"❌ 오류: 필요한 파일을 찾을 수 없습니다. (Tech Analysis 컬럼 포함 확인 필요) 경로를 확인하세요: {e}")
    exit()
except Exception as e:
    print(f"❌ 오류: 데이터 처리 중 예상치 못한 오류 발생: {e}")
    exit()


# --- 2. 정밀도 (Precision) 및 수익률 계산 ---

# BUY 성공: return_post_1d > 0
df_signals['is_correct'] = df_signals['return_post_1d'] > 0

# 1. 정밀도 (Precision) 계산
df_precision = df_signals.groupby('decision')['is_correct'].agg(['sum', 'count'])
df_precision['Precision (%)'] = (df_precision['sum'] / df_precision['count']) * 100
df_precision = df_precision.rename(columns={'sum': 'Correct Predictions', 'count': 'Total Signals'})


# 2. 평균 수익률 (Average Return) 계산
df_avg_return = df_signals.groupby('decision')[RETURN_COLS_SUMMARY].mean().mul(100).round(4)
df_avg_return = df_avg_return.rename(columns={'return_post_1d': 'Avg_Return_Post_1D (%)', 'return_post_2d': 'Avg_Return_Post_2D (%)'})


# --- 3. 최종 결과 출력 ---
print("\n" + "="*85)
print("🎯 최종 모델 Version 5 (5-Factor Hybrid) 성능 평가")
print("="*85)

print(f"적용 규칙: 5가지 조건 ALL-IN (GICS {MOST_SENSITIVE_GICS_SECTORS[0]} & Z > +{Z_SCORE_THRESHOLD} 등)")
print(f"총 BUY 시그널 수: {df_precision.iloc[0]['Total Signals']}개")

print("\n1. 정밀도 (Precision)")
print(df_precision[['Total Signals', 'Correct Predictions', 'Precision (%)']].to_markdown())
print("\n")

print("2. 평균 수익률 (Avg Return)")
print(df_avg_return.to_markdown())
print("="*85)

-> 1. 데이터 로드 및 모델 Version 5 규칙 적용...
❌ 오류: 데이터 처리 중 예상치 못한 오류 발생: Usecols do not match columns, columns expected but not found: ['momentum_rate', 'volume_ratio', 'rsi', 'gics_code']


NameError: name 'df_signals' is not defined

: 